## Development notebook

Here we check in pixel-by-pixel manner the differences between PIL and our re-implementation for drawing lines, circles etc.

In [ ]:
import sys, pathlib, os
sys.path.append(pathlib.Path.cwd().parents[1].__str__())
os.environ["IMAGE_UTILS_LOGLEVEL"] = "2"
os.environ["DEBUG_DRAW_LINE"] = "1"

from image_utils import image_draw_line, Color, image_draw_line, image_draw_polygon
from image_utils_pil import image_draw_line_pil, image_draw_polygon_pil
import numpy as np
from PIL import Image
# from contexttimer import Timer
import pandas as pd
from datetime import datetime

np.set_printoptions(linewidth=200)

%reload_ext autoreload
%autoreload 2

In [ ]:
def white(*size) -> np.ndarray:
    return (np.ones((*size, 3)) * Color.WHITE).astype(np.uint8)

def black(*size) -> np.ndarray:
    return (np.ones((*size, 3)) * Color.BLACK).astype(np.uint8)

def print_list(lst: list[int]):
    print("[", end="")
    for i, row in enumerate(lst):
        print("[", end="")
        print(", ".join([f"{x:3d}" for x in row]), end="")
        if i < len(lst) - 1:
            print("],\n ", end="")
        else:
            print("]", end="")
    print("]")

def image_display(image: np.ndarray):
    display(Image.fromarray(image)) # pylint: disable=all # noqa: F821


## Drawing lines

Turns out drawing lines is hard. The `size, p1, p2, thickness` tuple is used as-is in `image_draw_vs_pil_test.py` for the parametrized tests.

In [ ]:

fns = {
    "image_draw_line_pil": image_draw_line_pil,
    "image_draw_line": image_draw_line,
}

data = [
    ((20, 20), (5, 5), (15, 15), 10),
    ((20, 20), (10, 15), (15, 20), 20),
    ((15, 10), (5, 5), (10, 10), 20), # also not a normal diagonal.
    ((10, 10), (3, 3), (7, 9), 10), # not a normal diagonal.
    ((10, 10), (2, 2), (6, 6), 50), # mozaic pattern 3: 5 lines: odd ones have a skip_one (+/-1)
    ((10, 10), (3, 2), (6, 8), 10), # line slightly not aligned
    ((20 , 20 ), (10 , 10 ), (15 , 15 ), 1),
    ((20 , 20 ), (5  , 5  ), (15 , 15 ), 1),
    ((20 , 20 ), (2  , 7  ), (7  , 7  ), 20),
    ((250, 250), (100, 100), (100, 150), 20),
    ((10, 10), (2, 8), (7, 8), 20),
    ((30, 250), (10, 100), (10, 150), 20),
    ((15, 75), (5, 25), (5, 50), 50),
    ((10, 20), (5, 5), (5, 15), 1),
    ((20, 20), (10, 15), (10, 10), 10),
    ((10, 10), (2, 2), (6, 6), 10), # mozaic pattern 1: a single line without a skip_one
    ((480, 640), (100, 100), (300, 427), 10),
]
res = {k: [] for k in fns.keys()}

for item in data:
    size, p1, p2, thickness = item
    img = white(*size)
    for k, fn in fns.items():
        now = datetime.now()
        _ = fn(img, p1=p1, p2=p2, color=Color.BLACK, thickness=thickness)
        res[k].append((datetime.now() - now).total_seconds())

res = pd.DataFrame(res)
for col in [c for c in res.columns if c != "image_draw_line_pil"]:
    res[f"better_{col}"] = (res[col] / res["image_draw_line_pil"])
display(res)
display(res.mean().to_frame().T)


### Drawing polygons

In [ ]:

fns = {
    "image_draw_polygon_pil": image_draw_polygon_pil,
    "image_draw_polygon": image_draw_polygon,
}

data = [
    ((20, 20), [(10, 10), (10, 15), (15, 15), (15, 20), (20, 10)], 1),
    ((20, 20), [(10, 10), (10, 15), (15, 15), (15, 20), (20, 10)], 5),
    ((20, 20), [(10, 10), (10, 15), (15, 15), (15, 20), (20, 10)], 10),
]

res = {k: [] for k in fns.keys()}

for item in data:
    size, points, thickness = item
    img = white(*size)
    for k, fn in fns.items():
        now = datetime.now()
        _ = fn(img, points=points, color=Color.BLACK, thickness=thickness)
        res[k].append((datetime.now() - now).total_seconds())

res = pd.DataFrame(res)
for col in [c for c in res.columns if c != "image_draw_polygon_pil"]:
    res[f"better_{col}"] = (res[col] / res["image_draw_polygon_pil"])
display(res)
# display(res.mean().to_frame().T)
